# L67 · Edit Distance 从零实现 — Levenshtein 动态规划，WER 的数学基础

**学习目标**
- 理解编辑距离的三种操作：插入、删除、替换
- 从零实现 O(m×n) 动态规划算法
- 用它实现 WER（词错误率）核心计算
- 明白 ASR 评估为什么离不开它

## 为什么语音识别需要编辑距离？

给定参考文本 `ref = "the cat sat on the mat"` 和识别结果 `hyp = "the cat set on mat"`，
想知道 ASR "错了多少"。

**编辑距离** = 把 hyp 变成 ref 最少需要几步操作（插入/删除/替换）。
**词错误率 WER** = 词级别的编辑距离 / 参考文本词数。

**DP 状态**：`dp[i][j]` = 将 `a[:i]` 变成 `b[:j]` 的最小编辑次数。

**递推**：
```
if a[i-1] == b[j-1]:  dp[i][j] = dp[i-1][j-1]  # 字符相同，无需操作
else:
    dp[i][j] = 1 + min(
        dp[i-1][j],    # 删除 a[i-1]
        dp[i][j-1],    # 插入 b[j-1]
        dp[i-1][j-1]   # 替换 a[i-1] → b[j-1]
    )
```

In [ ]:
import numpy as np

## ✏️ 任务 1：实现字符级别编辑距离

In [ ]:
def edit_distance(a: str, b: str) -> int:
    """字符级别 Levenshtein 编辑距离，纯 Python 动态规划。"""
    m, n = len(a), len(b)
    # ✏️ TODO: 初始化 (m+1)×(n+1) 的 dp 表
    # dp[i][0] = i（删 i 个字符变成空串）
    # dp[0][j] = j（插 j 个字符得到 b[:j]）
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(m + 1):
        dp[i][0] = i
    for j in range(n + 1):
        dp[0][j] = j

    # ✏️ TODO: 填充 dp 表
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if a[i-1] == b[j-1]:
                dp[i][j] = dp[i-1][j-1]
            else:
                dp[i][j] = 1 + min(dp[i-1][j], dp[i][j-1], dp[i-1][j-1])

    return dp[m][n]

In [ ]:
# 验证
assert edit_distance('', '') == 0
assert edit_distance('abc', '') == 3
assert edit_distance('', 'abc') == 3
assert edit_distance('kitten', 'sitting') == 3   # 经典例子
assert edit_distance('hello', 'hello') == 0
print('✅ 字符级别编辑距离验证通过')

## ✏️ 任务 2：实现词级别 WER

In [ ]:
def wer(reference: str, hypothesis: str) -> float:
    """Word Error Rate = 词级别编辑距离 / 参考词数。"""
    ref_words = reference.lower().split()
    hyp_words = hypothesis.lower().split()
    if len(ref_words) == 0:
        return 0.0 if len(hyp_words) == 0 else float('inf')
    # ✏️ TODO: 对词列表应用编辑距离（把词当作字符即可）
    a, b = ref_words, hyp_words
    m, n = len(a), len(b)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(m + 1): dp[i][0] = i
    for j in range(n + 1): dp[0][j] = j
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if a[i-1] == b[j-1]:
                dp[i][j] = dp[i-1][j-1]
            else:
                dp[i][j] = 1 + min(dp[i-1][j], dp[i][j-1], dp[i-1][j-1])
    return dp[m][n] / len(ref_words)

In [ ]:
# 验证
assert wer('hello world', 'hello world') == 0.0
assert wer('hello world', 'hello') == 0.5        # 1 deletion / 2 words
assert abs(wer('a b c', 'a b c d') - 1/3) < 1e-9 # 1 insertion / 3 ref words

ref = 'the cat sat on the mat'
hyp = 'the cat set on mat'
print(f'ref: {ref}')
print(f'hyp: {hyp}')
print(f'WER = {wer(ref, hyp):.2%}')  # sat→set (S), deleted 'the' (D) = 2 errors / 6 words
print('✅ WER 验证通过')

## 对齐回溯：可视化每个错误

回溯 dp 表可以知道每个错误是 S（替换）/I（插入）/D（删除）。

In [ ]:
def alignment(ref_str, hyp_str):
    """打印词级别对齐路径，标注 M/S/I/D。"""
    a, b = ref_str.split(), hyp_str.split()
    m, n = len(a), len(b)
    dp = [[0]*(n+1) for _ in range(m+1)]
    for i in range(m+1): dp[i][0] = i
    for j in range(n+1): dp[0][j] = j
    for i in range(1, m+1):
        for j in range(1, n+1):
            if a[i-1] == b[j-1]: dp[i][j] = dp[i-1][j-1]
            else: dp[i][j] = 1 + min(dp[i-1][j], dp[i][j-1], dp[i-1][j-1])
    ops, i, j = [], m, n
    while i > 0 or j > 0:
        if i > 0 and j > 0 and a[i-1] == b[j-1]:
            ops.append(('M', a[i-1], b[j-1])); i -= 1; j -= 1
        elif i > 0 and j > 0 and dp[i][j] == 1 + dp[i-1][j-1]:
            ops.append(('S', a[i-1], b[j-1])); i -= 1; j -= 1
        elif j > 0 and dp[i][j] == 1 + dp[i][j-1]:
            ops.append(('I', '-', b[j-1])); j -= 1
        else:
            ops.append(('D', a[i-1], '-')); i -= 1
    print(f'{"Op":2s}  {"REF":12s}  {"HYP":12s}')
    print('-' * 30)
    for op, r, h in reversed(ops):
        marker = '  ' if op == 'M' else '**'
        print(f'{marker}{op:2s}  {r:12s}  {h:12s}{marker}')

alignment('the cat sat on the mat', 'the cat set on mat')

## 小结

| 概念 | 要记住的 |
|---|---|
| 编辑距离 | 3 种操作，DP 表 (m+1)×(n+1) |
| WER | = 词级编辑距离 / 参考词数，可以 > 1.0 |
| 回溯 | 从 dp[m][n] 往左上角走，还原对齐路径 |
| L68 | CTC 对齐——同样是"所有对齐路径"思想，但面对的是连续帧 |